# ClassifAI Webinar - Core Features

## ClassifAI Vectorisers

The Vectoriser Module and its classes provide a means of converting text to an embeddeding vector representation.

![ClassifAI Vectoriser](images/vectoriser.png)

In [ ]:
from classifai.vectorisers import HuggingFaceVectoriser 

demo_vectoriser = HuggingFaceVectoriser(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
demo_vectoriser.transform("Some string of text to embed")

In [ ]:
from classifai.vectorisers import GcpVectoriser

another_vectoriser = GcpVectoriser(
    project_id="---", 
    model_name="text-embedding-005"
    )

In [ ]:
a_vector = another_vectoriser.transform(["apple farmer on a farm", "fisherman on a boat"])


## ClassifAI VectorStore

The VectorStore is at the hear of the ClassifAI package, converting large amounts of coded data to a collection of embeddings. We can then use a the VectorStore to perform semantic search over the embedded collection, to find coded data most similar to a new uncoded sample.

![Vector space](./images/vectorstore_vectorspace.png)

In [ ]:
from classifai.indexers import VectorStore

demo_vectorstore = VectorStore(
    file_name = "/path to data",
    data_type = "csv",
    vectoriser = demo_vectoriser,
    output="demo_vectorstore"
    )


![vectorstore search](./images/vectorstore_search.png)

In [ ]:
from classifai.indexers.dataclasses import VectorStoreSearchInput

search_input = VectorStoreSearchInput({'id': [1], 'query': ["A carrot farmer on a farm"]})
search_input

In [ ]:
demo_vectorstore.search(search_input, n_results=5)

A VectorStore can be built containing upwards of 100K> rows of data which can take some time. Therefore, ClassifAI saves built VectorStores to filespace, to be reloaded and used at a later time

In [ ]:
reloaded_vectorstore = VectorStore.from_filespace(vectoriser=demo_vectoriser, folder_path="/path to first vectorstore folder")

In [ ]:
reloaded_vectorstore.search(search_input, n_results=5)

Finally, the VectorStore class has 2 more main methods beyond the search method 

- reverse_search() - which allows reverse lookups of the codes associated with each entry in the VectorStore

- embed() - convert text to embeddings using the VectorStore's associated Vectoriser.

In [ ]:
from classifai.indexers.dataclasses import VectorStoreReverseSearchInput

input_data_2 = VectorStoreReverseSearchInput({"id": ["1", "2"], "doc_label": ["101", "201"]})

reloaded_vectorstore.reverse_search(input_data_2)

In [ ]:
from classifai.indexers.dataclasses import VectorStoreEmbedInput

input_data_3 = VectorStoreEmbedInput(
    {
        "id": ["a", "b"],
        "text": [
            "The quick brown fox jumps over the lazy dog.",
            "Classifai is an amazing library for AI applications.",
        ],
    }
)

reloaded_vectorstore.embed(input_data_3)

| **VectorStore Method** | **Input Dataclass** | **Output Dataclass** |
|-------------------------------|-----------------------------|-----------------------------|
| `VectorStore.search()` | `VectorStoreSearchInput` | `VectorStoreSearchOutput` |
| `VectorStore.reverse_search()` | `VectorStoreReverseSearchInput` | `VectorStoreReverseSearchOutput` |
| `VectorStore.embed()` | `VectorStoreEmbedInput` | `VectorStoreEmbedOutput` |

## ClassifAI Servers

ClassifAI's server module allows you to easily deploy your VectorStore, setting up a Rest API for interfacing with the VectorStore over a network. This can be useful for deploying VectorStore capabilities as a Microservice, hostig on cloud platforms or generally making a VectorStore accessible to third parties.

![classifai vectorstore server](./images/vectorstore_server.png)


```python
##### first we load the vectoriser used in the vectorstore creation
from classifai.indexers import VectorStore
from classifai.servers import run_server
from classifai.vectorisers import HuggingFaceVectoriser

# Our embedding model is pulled down from HuggingFace, or used straight away if previously downloaded
# This also works with many different huggingface models!
vectoriser = HuggingFaceVectoriser(model_name="sentence-transformers/all-MiniLM-L6-v2")


#### now we can load the vectorstore back in without having to create it again
loaded_vectorstore = VectorStore.from_filespace("./DEMO/demo_vectorstore", demo_vectoriser)


#### look wow! you can search it straight away cause it was loaded back in

print(f"Test search {loaded_vectorstore.search('did the quick brown fox jump over the log?')}")


#### and finally, its easy to search your vectorstore via a restAPI service, just run:
run_server([loaded_vectorstore], endpoint_names=["my_vectorstore"])

# Look at https://0.0.0.0:8000/docs to see the Swagger API documentation and test in the browser

```

In [ ]:
!python demo_server_script.py

End slide.